In [1]:
# ================================
# 1. Install Dependencies
# ================================
!pip install unsloth transformers datasets scikit-learn accelerate

# ================================
# 2. Imports
# ================================
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from tqdm import tqdm

# ================================
# 3. Load Dataset
# ================================
file_path = "/content/OS_relations_fixed.json"

with open(file_path, "r") as f:
    data = json.load(f)

print("Total samples:", len(data))

# ================================
# 4. Split Few-shot + Test
# ================================
few_shot_examples = data[:3]   # ONLY FIRST 3
test_data = data[3:]          # REST

print("Few-shot examples:", len(few_shot_examples))
print("Test samples:", len(test_data))

# ================================
# 5. Load Model
# ================================
model_name = "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

# ================================
# 6. Build Few-shot Prompt
# ================================
def build_fewshot_prompt(example):

    prompt = "### Instruction:\n"
    prompt += example["instruction"] + "\n\n"

    prompt += "Choose ONLY from:\n"
    prompt += "used_for, based_on, implements, part_of, improves, no_relation\n\n"

    prompt += "### Examples:\n"

    # Add first 3 examples
    for ex in few_shot_examples:
        prompt += f"""Input: {ex['input']}
Output: {ex['output']}

"""

    # Add test input
    prompt += "### Now classify:\n"
    prompt += f"Input: {example['input']}\n"
    prompt += "Output:"

    return prompt

# ================================
# 7. Clean Prediction
# ================================
VALID_LABELS = ["used_for", "based_on", "implements", "part_of", "improves", "no_relation"]

def clean_prediction(pred):
    pred = pred.lower()
    for label in VALID_LABELS:
        if label in pred:
            return label
    return "no_relation"

# ================================
# 8. Prediction Function
# ================================
def predict(example):

    prompt = build_fewshot_prompt(example)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.0,
            do_sample=False
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    pred_raw = response.split("Output:")[-1].strip().split("\n")[0]

    pred = clean_prediction(pred_raw)

    return pred

# ================================
# 9. Run Evaluation
# ================================
y_true = []
y_pred = []

for example in tqdm(test_data):
    gt = example["output"].strip()
    pred = predict(example)

    y_true.append(gt)
    y_pred.append(pred)

# ================================
# 10. Metrics
# ================================
accuracy = accuracy_score(y_true, y_pred)

precision, recall, f1, _ = precision_recall_fscore_support(
    y_true, y_pred, average="weighted", zero_division=0
)

print("\n===== FEW-SHOT RESULTS =====")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

print("\n===== Classification Report =====")
print(classification_report(y_true, y_pred))

print("\n===== Confusion Matrix =====")
print(confusion_matrix(y_true, y_pred))

# ================================
# 11. Show Predictions
# ================================
for i in range(5):
    print("\n--- Example ---")
    print("Input :", test_data[i]["input"])
    print("GT    :", y_true[i])
    print("Pred  :", y_pred[i])

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 MB 11.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 32.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.8/403.8 kB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 102.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 224.

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/5.70G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

  0%|          | 0/1954 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=10) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
100%|██████████| 1954/1954 [1:31:46<00:00,  2.82s/it]


===== FEW-SHOT RESULTS =====
Accuracy : 0.4483
Precision: 0.4412
Recall   : 0.4483
F1 Score : 0.4352

===== Classification Report =====
              precision    recall  f1-score   support

    based_on       0.28      0.44      0.34        70
  implements       0.18      0.06      0.09        99
    improves       0.19      0.56      0.29        34
 no_relation       0.54      0.63      0.58       913
     part_of       0.43      0.28      0.34       557
    used_for       0.31      0.30      0.30       281

    accuracy                           0.45      1954
   macro avg       0.32      0.38      0.32      1954
weighted avg       0.44      0.45      0.44      1954


===== Confusion Matrix =====
[[ 31   0   3  30   1   5]
 [  5   6   1  50  21  16]
 [  3   0  19   5   3   4]
 [ 39   9  33 579 169  84]
 [ 25  12  29 254 158  79]
 [  9   6  14 150  19  83]]

--- Example ---
Input : Sentence: NTFS (New Technology File System): A modern file system used by Windows Entity1: File System